# ==============================================================================
# 📱 CUSTOMER RETENTION ANALYTICS: TELCO CHURN VIA BAGGING (RANDOM SUBSPACES)
# ==============================================================================
### Core Theoretical Principles:
Demonstrates **Random Subspaces Method**: sampling random subsets of features (`max_features=0.8`, `bootstrap_features=True`) to decorrelate individual trees on high-dimensional multi-service account profiles.


In [1]:
# STEP 1: ENTERPRISE ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Retention Analytics Environment ready.")


✅ STEP 1: Retention Analytics Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET HARMONIZATION
Loading Kaggle Telco Customer Churn dataset (7,043 subscriber accounts).


In [2]:
# STEP 2 & 3: INGESTION & AUDIT
data_path = 'data/customer_churn/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(data_path)

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

print(f"📊 Dataset Shape: {df.shape[0]} subscribers, {df.shape[1]} attributes")
print(f"Churn Distribution:\n{df['Churn'].value_counts(normalize=True).round(3)}")
df.head(3)


📊 Dataset Shape: 7043 subscribers, 21 attributes
Churn Distribution:
Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features and target `Churn`.


In [3]:
# STEP 4 & 5: SEGREGATION
X = df.drop(columns=['customerID', 'Churn']).copy()
y = (df['Churn'] == 'Yes').astype(int)

print(f"Subscriber Features: {X.shape[1]}")


Subscriber Features: 19


In [4]:
# STEP 6: STRATIFIED SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print(f"Train Subscribers: {X_train.shape[0]}")
print(f"Test Subscribers : {X_test.shape[0]}")


Train Subscribers: 5634
Test Subscribers : 1409


## ⚙️ STEP 7: PREPROCESSOR PIPELINE
Full ColumnTransformer for continuous tenures and categorical contract types.


In [5]:
# STEP 7: PREPROCESSOR
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_cols = [c for c in X.columns if c not in num_cols]

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
    ]), cat_cols)
])
print("✅ STEP 7: Preprocessor ready.")


✅ STEP 7: Preprocessor ready.


## ⚠️ STEP 8: BASELINE RANDOM SUBSPACES BAGGING FIT
Fitting BaggingClassifier with random feature subspaces.


In [6]:
# STEP 8: BASELINE FIT
pipe_bag = Pipeline([
    ('prep', preprocessor),
    ('bag', BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=6),
        n_estimators=100,
        max_features=0.8,
        bootstrap_features=True,
        random_state=42
    ))
])
pipe_bag.fit(X_train, y_train)

roc_bag = roc_auc_score(y_test, pipe_bag.predict_proba(X_test)[:, 1])
print(f"Random Subspaces Bagging ROC-AUC: {roc_bag * 100:.2f}%")


Random Subspaces Bagging ROC-AUC: 84.38%


## 🎯 STEP 9: TUNING HYPERPARAMETERS VIA 3-FOLD CV
Optimizing `max_features` and `max_samples`.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'bag__max_features': [0.7, 0.85],
    'bag__max_samples': [0.7, 0.85]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
grid = GridSearchCV(pipe_bag, param_grid=param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Retention Subspace Parameters: {grid.best_params_}")
print(f"🎯 Best 3-Fold CV ROC-AUC: {grid.best_score_*100:.2f}%")


🏆 Best Retention Subspace Parameters: {'bag__max_features': 0.85, 'bag__max_samples': 0.85}
🎯 Best 3-Fold CV ROC-AUC: 84.64%


## 📊 STEP 10 & 11: RETENTION REPORT & RESIDUALS
Evaluating retention intervention thresholds.


In [8]:
# STEP 10 & 11: EVALUATION REPORT
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("📋 RETENTION BAGGING EVALUATION REPORT:")
print(classification_report(y_test, y_pred, target_names=['Retained', 'Churn Risk']))
print(f"Test ROC-AUC Score : {roc_auc_score(y_test, y_prob):.4f}")
print(f"Test F1-Score      : {f1_score(y_test, y_pred):.4f}")


📋 RETENTION BAGGING EVALUATION REPORT:
              precision    recall  f1-score   support

    Retained       0.83      0.92      0.87      1035
  Churn Risk       0.68      0.48      0.57       374

    accuracy                           0.80      1409
   macro avg       0.76      0.70      0.72      1409
weighted avg       0.79      0.80      0.79      1409

Test ROC-AUC Score : 0.8443
Test F1-Score      : 0.5665


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & RETENTION SMOKE TEST
Deploying pipeline and validating multi-model bagged evaluation latency.


In [9]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/telco_churn_bagging_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_account = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_churn = loaded_pipe.predict(sample_account)[0]
pred_prob = loaded_pipe.predict_proba(sample_account)[0, 1]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n📱 LIVE SUBSCRIBER BAGGED RISK EVALUATION:")
print(f"   Retention Warning : {'🚨 HIGH CHURN RISK' if pred_churn == 1 else '🛡️ SECURE SUBSCRIBER'}")
print(f"   Churn Probability : {pred_prob * 100:.2f}%")
print(f"   Bagged Latency    : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/telco_churn_bagging_pipeline.joblib (305,659 bytes)



📱 LIVE SUBSCRIBER BAGGED RISK EVALUATION:
   Retention Warning : 🛡️ SECURE SUBSCRIBER
   Churn Probability : 2.32%
   Bagged Latency    : 38.591 ms (SLA < 2.0ms: CHECK)
